In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import mediapipe as mp

In [ ]:
dx = np.load('/Users/apple/Desktop/newData/NCX.npy')
dy = np.load('/Users/apple/Desktop/newData/NCY.npy')

In [3]:
class CNN_LSTM(nn.Module):
    def __init__(self,num_classes):
        super(CNN_LSTM,self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,16,kernel_size=3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), 
            nn.Flatten()
        )
        self.lstm = nn.LSTM(input_size=4608,hidden_size=64,batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64,num_classes)
        
    def forward(self,x):
        batch_size,seq_len,c,h,w = x.size()
        x = x.view(batch_size * seq_len, c, h, w)
        cnn_out = self.cnn(x)
        cnn_out = cnn_out.view(batch_size, seq_len, -1)
        lstm_out, _ = self.lstm(cnn_out)
        last_output = lstm_out[:, -1, :]
        out = self.dropout(last_output)
        out = self.fc(out) 
        
        return out


In [4]:
num_classes = 5  

In [ ]:
from sklearn.model_selection import train_test_split
BATCH_SIZE = 1

X_train,X_test,y_train,y_test = train_test_split(dx,dy,test_size=0.2,shuffle=True)
X_test,X_val,y_test,y_val = train_test_split(X_test,y_test,test_size=0.5,shuffle=True)

train_dataset = BadDataset(X_train,y_train)
test_dataset = BadDataset(X_test,y_test)
val_dataset = BadDataset(X_val,y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = Model(num_classes).to(device)
model = CNN_LSTM(num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


In [ ]:
from tqdm import tqdm_notebook
EPOCHS = 5

epoch_bar = tqdm_notebook(desc='training routine', 
                          total=EPOCHS,
                          position=0)

train_bar = tqdm_notebook(desc='split=train',
                          total=len(train_loader), 
                          position=1, 
                          leave=True)

val_bar = tqdm_notebook(desc='split=val',
                        total=len(val_loader), 
                        position=1, 
                        leave=True)


for epoch in range(EPOCHS): 
    total_loss = 0
    total_acc = 0
    running_loss = 0.0
    running_acc = 0.0
    model.train()
    for idx,X in enumerate(train_loader):
        src, trg = X[0].to(device), X[1].to(device)
        src = src.unsqueeze(0)
        src = src.permute(0,2,1,3,4)
        
        optimizer.zero_grad()
        output = model(src)  

        loss = criterion(output, trg)
        acc = int(torch.argmax(output,dim=1)==trg)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += acc
        
        running_loss += (loss.item() - running_loss) / (idx + 1)
        running_acc += (acc - running_acc) / (idx + 1)
        
        train_bar.set_postfix(loss=running_loss, 
                                acc=running_acc, 
                                epoch=epoch)
        train_bar.update()
        
    loss, acc = total_loss / len(train_loader), total_acc / len(train_loader)
    avg_loss = loss / len(train_loader)
    
    model.eval()
    total_loss = 0
    total_acc = 0
    running_loss = 0.
    running_acc = 0.
    with torch.no_grad():
        for idx,X in enumerate(val_loader):
            src, trg = X[0].to(device), X[1].to(device)
            src = src.unsqueeze(0)
            src = src.permute(0,2,1,3,4)
            output = model(src)  

            loss = criterion(output, trg)
            acc = int(torch.argmax(output,dim=1)==trg)

            total_loss += loss.item()
            total_acc += acc
            
            running_loss += (loss.item() - running_loss) / (idx + 1)
            running_acc += (acc - running_acc) / (idx + 1)

            val_bar.set_postfix(loss=running_loss, 
                                acc=running_acc, 
                                epoch=epoch)
            val_bar.update()
        
    epoch_bar.update()
    train_bar.n = 0
    val_bar.n = 0
    

In [ ]:
test_bar = tqdm_notebook(desc='split=test',
                          total=len(test_loader), 
                          position=1, 
                          leave=True)

model.eval()
total_loss = 0
total_acc = 0
running_loss = 0.
running_acc = 0.
with torch.no_grad():
    for idx,X in enumerate(test_loader):
        src, trg = X[0].to(device), X[1].to(device)
        src = src.unsqueeze(0)
        src = src.permute(0,2,1,3,4)
        output = model(src)  

        loss = criterion(output.view(-1, output.shape[-1]), trg.view(-1))
        acc = int(torch.argmax(output,dim=1)==trg)

        total_loss += loss.item()
        total_acc += acc
                
        running_loss += (loss.item() - running_loss) / (idx + 1)
        running_acc += (acc - running_acc) / (idx + 1)

        test_bar.set_postfix(loss=running_loss, 
                                acc=running_acc)
        test_bar.update()


In [ ]:
torch.save(model,'/Users/apple/Desktop/newData/NCModel.pt')
torch.save(model.state_dict(),'/Users/apple/Desktop/newData/NCModel_dict.pt')

Inference

In [ ]:
model = YOLO('/Users/apple/Downloads/player_bbox4.pt')
model2 = torch.load('/Users/apple/Desktop/newData/NCModel.pt')
model2.load_state_dict(torch.load('/Users/apple/Desktop/newData/NCModel_dict.pt'))
device = torch.device("cuda" if torch.cuda.is_available() else "mps")
model2 = model2.to(device)

In [ ]:

# COCO limb pairs
LIMB_PAIRS = [
    (5, 7), (7, 9),     # Left arm
    (6, 8), (8, 10),    # Right arm
    (11, 13), (13, 15), # Left leg
    (12, 14), (14, 16), # Right leg
    (5, 6), (11, 12),   # Shoulders, hips
    (0, 1), (0, 2), (1, 3), (2, 4)  # Head
]

COCO_MAP = {
    0: 0, 1: 2, 2: 5, 3: 7, 4: 8,
    5: 11, 6: 12, 7: 13, 8: 14, 9: 15,
    10: 16, 11: 23, 12: 24, 13: 25,
    14: 26, 15: 27, 16: 28
}

def mediapipe_to_coco(mp_landmarks, width, height):
    keypoints_coco = []
    for coco_id in range(17):
        mp_id = COCO_MAP[coco_id]
        lm = mp_landmarks[mp_id]
        x = int(lm.x * width)
        y = int(lm.y * height)
        keypoints_coco.append((x, y, lm.visibility))
    return keypoints_coco

def draw_limb_heatmap(keypoints, image_size, sigma=8):
    h, w = image_size
    heatmap = np.zeros((h, w), dtype=np.float32)

    for (start_idx, end_idx) in LIMB_PAIRS:
        x1, y1, s1 = keypoints[start_idx]
        x2, y2, s2 = keypoints[end_idx]
        if s1 < 0.1 or s2 < 0.1:
            continue
        canvas = np.zeros((h, w), dtype=np.uint8)
        cv2.line(canvas, (x1, y1), (x2, y2), color=255, thickness=2*sigma)
        canvas = cv2.GaussianBlur(canvas, (0, 0), sigmaX=sigma, sigmaY=sigma)
        canvas = canvas.astype(np.float32) / 255.0
        heatmap = np.maximum(heatmap, canvas)
    return heatmap



In [ ]:

import os
import time
start = time.time()

directory = '/Users/apple/Downloads/TrackNetV2/Professional/match4/video/2_14_17.mp4'

dic = {0:'backhand',1:'block',2:'drop',3:'serve',4:'smash'}

pose = mp.solutions.pose.Pose(min_detection_confidence=0.7,min_tracking_confidence=0.7)

i=0
j=[]

video = cv2.VideoCapture(directory)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')



frame_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = int(video.get(cv2.CAP_PROP_FPS))
no_of_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
secs = int(no_of_frames/fps)


out = cv2.VideoWriter('/Users/apple/Downloads/output5.mp4', fourcc, fps, (1280,720))


#every 6th frame
i=0
k=0
frames = []
while True:
    success, frame = video.read()
    
    if not success:
        print("error")
        break
        

    result = model(frame,conf=0.60)
    box = result[0].boxes.xyxy
    
        
    cv2.rectangle(frame, (int(box[0][0]), int(box[0][1])), (int(box[0][2]), int(box[0][3])), (255,0,0), 2)
    if box.shape[0]==2:
        cv2.rectangle(frame, (int(box[1][0]), int(box[1][1])), (int(box[1][2]), int(box[1][3])), (0,255,0), 2)

    if i%6==0:
        Box = getMidPoint(result[0].boxes.xyxy[0])
        z=[]        
        results = pose.process(frame[Box[1]-125:Box[1]+125,Box[0]-125:Box[0]+125])
    
        if results.pose_landmarks !=None:
            keypoints_coco = mediapipe_to_coco(results.pose_landmarks.landmark, 50, 50)
            heatmap = draw_limb_heatmap(keypoints_coco, (50, 50))
            z.append(heatmap)
            z = np.asarray(z)
            frames.append(z)
        
    if len(frames)>=10:
        tfr = np.asarray(frames[-10:])
        tfr = tfr.reshape((1,10,50,50))
        tfr = torch.tensor(tfr).to(device)
        tfr = tfr.unsqueeze(0)
        tfr = tfr.permute(0,2,1,3,4)
        output = model2(tfr)
        o = torch.argmax(output,dim=1)
        print(dic[o.item()])
        cv2.putText(frame,dic[o.item()], (int(box[0][0]), int(box[0][1])),cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),1,2)

    if len(frames)>=30:
        frames = frames[10:]
    
    out.write(frame)
    
    i+=1

video.release()
out.release()
cv2.destroyAllWindows()
print(f"Time: {time.time()-start}")
